In [1]:
%load_ext autoreload
%autoreload 2

In [1]:
# HACK: this ensure we can import relative modules
import sys
import os

sys.dont_write_bytecode = True

if (module_path := os.path.abspath('..')) not in sys.path:
    sys.path.append(module_path)

In [3]:
import pathlib
import os
import glob
from typing import Final
import re
from collections import defaultdict


import polars as pl

from analysis import data_pipeline
from analysis.string_utils import camel_to_snake

DATA_PATH: Final[pathlib.Path] = pathlib.Path(os.getcwd()).parent / "data"

# Initial analysis
Understanding data:
- [TLC Rules](https://www.nyc.gov/site/tlc/about/tlc-rules.page)
    - [Chapter 80 – Medallion Taxicab, SHL and For-Hire Drivers](https://www.nyc.gov/assets/tlc/downloads/pdf/rule_book_current_chapter_80.pdf) (after 25/10/2016)
- [Old TLC Rules](https://www.nyc.gov/site/tlc/about/tlc-rules.page)
    - [Chapter 54 – Medallion Taxicab and SHL Drivers](https://www.nyc.gov/assets/tlc/downloads/pdf/rule_book_current_chapter_54.pdf) (until 25/10/2016)
- [Trip Record User Guide](https://www.nyc.gov/assets/tlc/downloads/pdf/trip_record_user_guide.pdf)

## Source of Data
TPEP (Taxicab Technology System) and LPEP (Street Hail Livery Technology System) are the electronic hardware and software systems installed in New York City's for-hire vehicles to collect trip data, process payments, and provide information to both drivers and passengers.

### TPEP (Taxicab Technology System)
TPEP is specifically used in yellow Taxicabs, also known as yellow taxis

### LPEP (Street Hail Livery Technology System)
LPEP is the equivalent system used in Street Hail Liveries, also known as green taxis

In [4]:
files = defaultdict(lambda: defaultdict(lambda: defaultdict(str)))
pattern = re.compile(r"^.*?(?P<taxi>green|yellow).*?(?P<year>\d{4})-(?P<month>\d{2})\.parquet$")
for f in glob.glob(str(DATA_PATH / "**" / "*.parquet"), recursive=True):
    if match := pattern.match(f):
        taxi, year, month = match.groups()
        files[taxi][year][month] = f

for year in files["yellow"]:
    for month in files["yellow"][year]:
        columns_by_taxi = defaultdict(set) 
        for taxi in files:
            if files[taxi][year][month]:
                df = pl.read_parquet(files[taxi][year][month], n_rows=1)
                columns_by_taxi[taxi] = set(df.schema.keys())
        common_columns = set.intersection(*columns_by_taxi.values())
        print("-" * 40)
        print(f"{year}/{month}")
        print(f"Common columns: {sorted(common_columns)}")
        print(f"Common columns count: {len(common_columns)}")
        for taxi in columns_by_taxi:
            unique_columns = columns_by_taxi[taxi] - common_columns
            label = taxi.capitalize()
            print(f"{label} only columns: {sorted(unique_columns)}")
            print(f"{label} only columns count: {len(unique_columns)}")
            print(f"{label} only total columns: {len(columns_by_taxi[taxi])}")
        total_columns = sum(len(cols) for cols in [*[col - common_columns for col in columns_by_taxi.values()], common_columns])
        print(f"Total columns: {total_columns}")

----------------------------------------
2009/01
Common columns: ['End_Lat', 'End_Lon', 'Fare_Amt', 'Passenger_Count', 'Payment_Type', 'Rate_Code', 'Start_Lat', 'Start_Lon', 'Tip_Amt', 'Tolls_Amt', 'Total_Amt', 'Trip_Distance', 'Trip_Dropoff_DateTime', 'Trip_Pickup_DateTime', 'mta_tax', 'store_and_forward', 'surcharge', 'vendor_name']
Common columns count: 18
Yellow only columns: []
Yellow only columns count: 0
Yellow only total columns: 18
Total columns: 18
----------------------------------------
2009/12
Common columns: ['End_Lat', 'End_Lon', 'Fare_Amt', 'Passenger_Count', 'Payment_Type', 'Rate_Code', 'Start_Lat', 'Start_Lon', 'Tip_Amt', 'Tolls_Amt', 'Total_Amt', 'Trip_Distance', 'Trip_Dropoff_DateTime', 'Trip_Pickup_DateTime', 'mta_tax', 'store_and_forward', 'surcharge', 'vendor_name']
Common columns count: 18
Yellow only columns: []
Yellow only columns count: 0
Yellow only total columns: 18
Total columns: 18
----------------------------------------
2010/01
Common columns: ['dropoff

#### Insights

##### Data Origin
While the TPEP or LPEP is inoperative, drivers must file an incident report with the providers and maintain a written trip log for the duration they are authorized to operate without the system. Therefore, we have data directly from the systems as well as data entered manually. Know that is important for data sanitization.

##### Dictionaries
We have only 2025 and 2015 data dictionaries an both have few differences.
###### Yellow
Yellow has added a new type of `RatecodeID`, `99 = Null/unknown`, and changed `VendorID` from `1=Creative Mobile Technologies, LLC; 2=VeriFone Inc.` to `1=Creative Mobile Technologies, LLC; 2=Curb Mobility, LLC; 6=Myle Technologies Inc; 7=Helix`.
###### Green
Green has added a new type of `RatecodeID`, `99 = Null/unknown`, and changed `VendorID` from `1=Creative Mobile Technologies, LLC; 2=VeriFone Inc.` to `1=Creative Mobile Technologies, LLC; 2=Curb Mobility, LLC; 6=Myle Technologies Inc`.
###### Conclusions
Both systems seems almost the same with few differences, so we can merge both to solve small files problem (file size < 64 MB)

**About `VendorID`:** This is not a issue of the data, at 2015 VeriFone acquired Curb, then brough back Curb at 2018, so they are the same entity.

**About `RatecodeID`:** This seems like an improvement for data entered manually, we can use as a standard value when null data is found.

##### Data divergence
As we can see, the data has evolved, so we will focus only on data from 2011/01 onwards. Earlier data requires more effort because it uses geospatial data, and we would have to process it to populate LocationsIDs. This is a feature by himself and will not be used in challenge.

In [5]:
taxi_zone_lookup = pl.scan_csv(DATA_PATH / "taxi_zone_lookup.csv")\
    .rename(camel_to_snake)\
    .with_columns(
        pl.col("location_id").cast(pl.Int32),
        pl.col("borough").cast(pl.Categorical),
        pl.col("zone").cast(pl.Categorical),
        pl.col("service_zone").cast(pl.Categorical)
    )

In [6]:
import polars.selectors as cs

tz_df = taxi_zone_lookup.collect()
(
    tz_df
    .select(pl.all().n_unique().name.suffix("_unique"), pl.all().len().name.suffix("_total"))
    .with_columns((cs.ends_with("_unique") / cs.ends_with("_total")).name.suffix("_ratio"))
)
# NOTE: This could be normalized to 3NF

location_id_unique,borough_unique,zone_unique,service_zone_unique,location_id_total,borough_total,zone_total,service_zone_total,location_id_unique_ratio,borough_unique_ratio,zone_unique_ratio,service_zone_unique_ratio
u32,u32,u32,u32,u32,u32,u32,u32,f64,f64,f64,f64
265,8,262,5,265,265,265,265,1.0,0.030189,0.988679,0.018868


<details>
<summary><b>Data Dictionary</b></summary>
</br>

| Field Name | Description |
| --- | --- |
| VendorID | A code indicating the TPEP provider that provided the record. <ul style="list-style-type: none"><li>1 = Creative Mobile Technologies, LLC</li><li>2 = Curb Mobility, LLC</li><li>6 = Myle Technologies Inc</li><li>7 = Helix</li> |
| tpep_pickup_datetime | The date and time when the meter was engaged. |
| tpep_dropoff_datetime | The date and time when the meter was disengaged. |
| passenger_count | The number of passengers in the vehicle. |
| trip_distance | The elapsed trip distance in miles reported by the taximeter. |
| RatecodeID | The final rate code in effect at the end of the trip. <ul style="list-style-type: none"><li>1 = Standard rate</li><li>2 = JFK</li><li>3 = Newark</li><li>4 = Nassau or Westchester</li><li>5 = Negotiated fare</li><li>6 = Group ride</li><li>99 = Null/unknown</li></ul> |
| store_and_fwd_flag | This flag indicates whether the trip record was held in vehicle memory before sending to the vendor, aka “store and forward,” because the vehicle did not have a connection to the server.<ul style="list-style-type: none"><li>Y = store and forward trip</li><li>N = not a store and forward trip</li></ul> |
| PULocationID | TLC Taxi Zone in which the taximeter was engaged. |
| DOLocationID | TLC Taxi Zone in which the taximeter was disengaged. |
| payment_type | A numeric code signifying how the passenger paid for the trip.<ul style="list-style-type: none"><li>0 = Flex Fare trip</li><li>1 = Credit card</li><li>2 = Cash</li><li>3 = No charge</li><li>4 = Dispute</li><li>5 = Unknown</li><li>6 = Voided trip</li></ul> |
| fare_amount | The time-and-distance fare calculated by the meter. For additional information on the following columns, see https://www.nyc.gov/site/tlc/passengers/taxi-fare.page extra Miscellaneous extras and surcharges. |
| mta_tax | Tax that is automatically triggered based on the metered rate in use. |
| tip_amount | Tip amount – This field is automatically populated for credit card tips. Cash tips are not included. |
| tolls_amount | Total amount of all tolls paid in trip. |
| improvement_surcharge | Improvement surcharge assessed trips at the flag drop. The improvement surcharge began being levied in 2015. |
| total_amount | The total amount charged to passengers. Does not include cash tips. |
| congestion_surcharge | Total amount collected in trip for NYS congestion surcharge. |
| airport_fee | For pick up only at LaGuardia and John F. Kennedy Airports. |
| cbd_congestion_fee | Per-trip charge for MTA's Congestion Relief Zone starting Jan. 5, 2025. |

</details>

Source: [Yellow Trips Data Dictionary](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf)

<details>
<summary><b>Data Dictionary</b></summary>

| Field Name | Description |
| --- | --- |
| VendorID | A code indicating the LPEP provider that provided the record. <ul style="list-style-type: none"><li>1 = Creative Mobile Technologies, LLC</li><li>2 = Curb Mobility, LLC</li><li>6 = Myle Technologies Inc</li></ul> |
| lpep_pickup_datetime | The date and time when the meter was engaged. |
| lpep_dropoff_datetime | The date and time when the meter was disengaged. |
| store_and_fwd_flag |  This flag indicates whether the trip record was held in vehicle memory before sending to the vendor, aka “store and forward,” because the vehicle did not have a connection to the server. |
| RatecodeID | The final rate code in effect at the end of the trip. <ul style="list-style-type: none"><li>1 = Standard rate</li><li>2 = JFK</li><li>3 = Newark</li><li>4 = Nassau or Westchester</li><li>5 = Negotiated fare</li><li>6 = Group ride</li><li>99 = Null/unknown</li></ul> |
| PULocationID | TLC Taxi Zone in which the taximeter was engaged. |
| DOLocationID | TLC Taxi Zone in which the taximeter was disengaged. |
| passenger_count | The number of passengers in the vehicle. |
| trip_distance | The elapsed trip distance in miles reported by the taximeter. |
| fare_amount | The time-and-distance fare calculated by the meter. For additional information on the following columns, see https://www.nyc.gov/site/tlc/passengers/taxi-fare.page extra Miscellaneous extras and surcharges. |
| mta_tax | Tax that is automatically triggered based on the metered rate in use. |
| tip_amount | Tip amount – This field is automatically populated for credit card tips. Cash tips are not included. |
| tolls_amount | Total amount of all tolls paid in trip. |
| improvement_surcharge | Improvement surcharge assessed trips at the flag drop. The improvement surcharge began being levied in 2015. |
| total_amount | The total amount charged to passengers. Does not include cash tips. |
| payment_type | A numeric code signifying how the passenger paid for the trip. <ul style="list-style-type: none"><li>0 = Flex Fare trip</li><li>1 = Credit card</li><li>2 = Cash</li><li>3 = No charge</li><li>4 = Dispute</li><li>5 = Unknown</li><li>6 = Voided trip</li></ul>
| trip_type | A code indicating whether the trip was a street-hail or a dispatch that is automatically assigned based on the metered rate in use but can be altered by the driver.<ul style="list-style-type: none"><li>1 = Street-hail</li><li>2 = Dispatch</li></ul> |
| congestion_surcharge | Total amount collected in trip for NYS congestion surcharge. |
| cbd_congestion_fee | Per-trip charge for MTA's Congestion Relief Zone starting Jan. 5, 2025. |

</details>

Source: [Green Trips Data Dictionary](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_green.pdf)

<details>
<summary>ehail_fee</summary>

This was not included at dictionary, so we can consider that as a some sort of mistype or catalog issue, then we will add to main dictionary. 

```markdown
The New York City Taxi and Limousine Commission (TLC) has implemented a program allowing licensed E-hail providers to offer upfront pricing in all participating yellow and green taxicabs. In addition to metered fares, passengers riding in Medallion taxicabs and Street Hail Liveries (SHLs) will now have the option to receive binding fare quotes when completing an E-Hail request.
See below for a list of current participants:

- Myle Technologies Inc. 
- Curb Mobility
- Arro

Note: This is an active list. Therefore, it will be updated as new participants join.

- If you request a trip through the E-Hail app, the meter will not be turned on for the duration of the trip.
- Each E-Hail company sets their own rates and will give you a price prior to sending a trip request.
- If you received a binding fare quote, you will pay the fare that you accepted. E-Hail companies are still allowed to send you metered taxis, instead of offering a binding fare quote. Consult the E-Hail company if you are unsure which type of trip you are requesting.
- The Port Authority does not currently allow taxis to be E-Hailed at the airport. However, you can be dropped off at the airport.
- The upfront fare quote should include all charges, but the fare may change if you change your destination or if there are any unexpected tolls or taxes. If you feel you have been overcharged file a complaint with 311.
- Each app will have an option to tip the driver. As always, you can tip your driver in cash.
```

Source: [Taxi Fare > E-Hail](https://www.nyc.gov/site/tlc/passengers/taxi-fare.page)
</details>

In [7]:
files = glob.glob(str(DATA_PATH / "*" / "*_tripdata_2023*.parquet"))
df: pl.DataFrame = data_pipeline.run_pipeline(files, "etl", "tlc", zone_lookup=taxi_zone_lookup)
print(f"DF size in memory: {df.estimated_size('gb') :.2f} GB")
df.head()

DF size in memory: 2.28 GB


congestion_surcharge,dropoff_borough,dropoff_datetime,dropoff_location_id,dropoff_service_zone,dropoff_zone,extra,fare_amount,improvement_surcharge,mta_tax,origin,origin_id,passenger_count,payment_type,payment_type_id,pickup_borough,pickup_datetime,pickup_location_id,pickup_service_zone,pickup_zone,ratecode,ratecode_id,store_and_foward,tip_amount,tolls_amount,total_amount,trip_distance_mi,trip_type,trip_type_id,vendor,vendor_id,airport_fee
f64,cat,datetime[μs],i16,cat,cat,f64,f64,f64,f64,cat,i8,i8,cat,i8,cat,datetime[μs],i16,cat,cat,cat,i8,cat,f64,f64,f64,f64,cat,i8,cat,i8,f64
0.0,"""Queens""",2023-01-20 19:03:10,138,"""Airports""","""LaGuardia Airport""",7.5,17.7,1.0,0.5,"""LPEP""",2,1,"""Cash""",2,"""Queens""",2023-01-20 18:53:31,260,"""Boro Zone""","""Woodside""","""Standard rate""",1,"""N""",0.0,0.0,26.7,4.11,"""Street-hail""",1,"""Curb Mobility, LLC""",2,null
2.75,"""Queens""",2023-01-11 09:15:49,146,"""Boro Zone""","""Long Island City/Queens Plaza""",2.75,36.6,1.0,1.5,"""LPEP""",2,1,"""Cash""",2,"""Manhattan""",2023-01-11 08:33:23,74,"""Boro Zone""","""East Harlem North""","""Standard rate""",1,"""N""",0.0,0.0,40.85,8.0,"""Street-hail""",1,"""Creative Mobile Technologies, …",1,null
0.0,"""Queens""",2023-01-25 13:32:34,134,"""Boro Zone""","""Kew Gardens""",0.0,8.6,1.0,0.5,"""LPEP""",2,1,"""Credit card""",1,"""Queens""",2023-01-25 13:25:52,134,"""Boro Zone""","""Kew Gardens""","""Standard rate""",1,"""N""",0.0,0.0,10.1,1.1,"""Street-hail""",1,"""Curb Mobility, LLC""",2,null
2.75,"""Manhattan""",2023-01-24 17:20:56,263,"""Yellow Zone""","""Yorkville West""",2.5,9.3,1.0,0.5,"""LPEP""",2,1,"""Credit card""",1,"""Manhattan""",2023-01-24 17:13:45,75,"""Boro Zone""","""East Harlem South""","""Standard rate""",1,"""N""",2.09,0.0,18.14,1.31,"""Street-hail""",1,"""Curb Mobility, LLC""",2,null
2.75,"""Manhattan""",2023-01-26 11:45:35,161,"""Yellow Zone""","""Midtown Center""",0.0,21.2,1.0,0.5,"""LPEP""",2,6,"""Credit card""",1,"""Manhattan""",2023-01-26 11:22:22,43,"""Yellow Zone""","""Central Park""","""Standard rate""",1,"""N""",5.09,0.0,30.54,2.59,"""Street-hail""",1,"""Curb Mobility, LLC""",2,null


In [8]:
sample = df.group_by("store_and_foward", "payment_type_id", maintain_order=True).agg(
    pl.col("*").top_k_by("store_and_foward", k=1)
).explode(pl.exclude("store_and_foward", "payment_type_id"))
sample.write_csv("sample.csv", separator=";")
del sample

In [ ]:
# NOTE: This is the answer to question 1.
(
    df
    .filter(
        (pl.col("origin_id") == 1) 
        & (pl.col("pickup_datetime").dt.year() == 2023) 
        & (pl.col("pickup_datetime").dt.month().is_between(1, 5))
        & (pl.col("total_amount") >= 0)
    )
    .select(
        "vendor_id", "passenger_count", "total_amount", "pickup_datetime", "dropoff_datetime"
    )
    .group_by(pl.col("pickup_datetime").dt.month())
    .agg(
        pl.col("total_amount").mean().alias("avg_total_amount"),
    )
    .sort(pl.col("pickup_datetime"))
)

pickup_datetime,avg_total_amount
i8,f64
1,27.418547
2,27.342784
3,28.258522
4,28.743893
5,29.425365


In [29]:
# NOTE: This is the answer to question 2.
(   
    df
    .filter(
        (pl.col("pickup_datetime").dt.year() == 2023) 
        & (pl.col("pickup_datetime").dt.month() == 5)
        & (pl.col("total_amount") >= 0)
    )
    .select(
        "vendor_id", "passenger_count", "total_amount", "pickup_datetime", "dropoff_datetime"
    )
    .with_columns(
        (pl.col("dropoff_datetime") - pl.col("pickup_datetime")).alias("trip_duration")
    )
    .group_by(pl.col("pickup_datetime").dt.hour())
    .agg(
        pl.col("passenger_count").mean().alias("avg_passenger_count"),
    )
    .sort(pl.col("pickup_datetime"))
)

pickup_datetime,avg_passenger_count
i8,f64
0,1.411727
1,1.42269
2,1.437846
3,1.436196
4,1.391559
…,…
19,1.368872
20,1.380983
21,1.401559
